In [ ]:
import os
import torch
import pandas as pd
from data.datasets import CropDataset
from models.resnet import build_resnet50
from PIL import Image
import numpy as np

In [ ]:
DATA_DIR = ""

In [ ]:
DATASET_DIR = ""

In [ ]:
CKPT_DIR=""

In [ ]:
DATASET_PATHS={
    ('nigeria','2022'): os.path.join(DATASET_DIR, "nigeria_2022.csv"),
    ('zambia','2023'):os.path.join(DATASET_DIR, "zambia_2023.csv"),
    ('zambia','2024'):os.path.join(DATASET_DIR, "zambia_2024.csv"),
    ('zimbabwe','2024'):os.path.join(DATASET_DIR, "zimbabwe_2024.csv"),
}

In [ ]:
CKPT_PATHS={
    ('nigeria','2022'): os.path.join(CKPT_DIR, ""),
    ('zambia','2023'):os.path.join(CKPT_DIR, ""),
    ('zambia','2024'):os.path.join(CKPT_DIR, ""),
    ('zimbabwe','2024'):os.path.join(CKPT_DIR, ""),
}

In [ ]:
def gen_predictions(country, year, fold):
    csv_path=DATASET_PATHS[(country,year)]
    df = pd.read_csv(csv_path)
    ckpt_path=CKPT_PATHS[(country,year)]
    model = build_resnet50()
    state_dict = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(state_dict)
    model.cuda()
    model.eval()
    for index, row in df.iterrows():
        if row['fold'] != fold: 
            continue
        image = Image.open(os.path.join(DATA_DIR, f"q_field_photo/q_field_photo_{row['cce_id']}.jpg"))
        transform = CropDataset.build_transform(False, 224)
        image = transform(image).cuda()
        image = image.unsqueeze(0)
        with torch.no_grad():
            output = model(image)
        col_name = f'pred_fold{fold}'
        if col_name not in df.columns:
            df[col_name] = np.nan
        df.loc[index, col_name] = output.item()
    return df

In [ ]:
for country, year in DATASET_PATHS:
    for fold in range(5):
        df = gen_predictions(country, year)
        df.to_csv(os.path.join(DATASET_DIR, ""), index=False)